# **Importing libraries and loading data**
###For analysis Pandas, Numpy will be used, 

Data was downloadedfrom Kaggle, which was spotifytoptracks.csv

For easier understanding, miliseconds were changed to minutes.

In [55]:
import pandas as pd
import numpy as np

df = pd.read_csv("spotifytoptracks.csv")
df['duration_min'] = round(df['duration_ms'] / (1000 * 60), 2)
df = df.drop('duration_ms', axis=1)
df.head()

,Unnamed: 0,artist,album,track_name,track_id,energy,danceability,key,loudness,acousticness,speechiness,instrumentalness,liveness,valence,tempo,genre,duration_min
0,0,The Weeknd,After Hours,Blinding Lights,0VjIjW4GlUZAMYd2vXMi3b,0.730,0.514,1,-5.934,0.00146,0.0598,0.000095,0.0897,0.334,171.005,R&B/Soul,3.33
1,1,Tones And I,Dance Monkey,Dance Monkey,1rgnBhdG2JDFTbYkYRZAku,0.593,0.825,6,-6.401,0.68800,0.0988,0.000161,0.1700,0.540,98.078,Alternative/Indie,3.50
2,2,Roddy Ricch,Please Excuse Me For Being Antisocial,The Box,0nbXyq5TXYPCO7pr3N8S4I,0.586,0.896,10,-6.687,0.10400,0.0559,0.000000,0.7900,0.642,116.971,Hip-Hop/Rap,3.28
3,3,SAINt JHN,Roses (Imanbek Remix),Roses - Imanbek Remix,2Wo6QQD1KMDWeFkkjLqwx5,0.721,0.785,8,-5.457,0.01490,0.0506,0.004320,0.2850,0.894,121.962,Dance/Electronic,2.94
4,4,Dua Lipa,Future Nostalgia,Don't Start Now,3PfIrDoz19wz7qK7tYeu62,0.793,0.793,11,-4.521,0.01230,0.0830,0.000000,0.0951,0.679,123.950,Nu-disco,3.05


# **Data cleaning**

### Missing values and Duplicates:
Checking for empty values and duplicates, by using sum  to count duplicates and isnull count to count missing values. 

In [56]:
print(df.isnull().sum())
print("Duplicate values: ", df.duplicated().sum())

Unnamed: 0          0
artist              0
album               0
track_name          0
track_id            0
energy              0
danceability        0
key                 0
loudness            0
acousticness        0
speechiness         0
instrumentalness    0
liveness            0
valence             0
tempo               0
genre               0
duration_min        0
dtype: int64
Duplicate values:  0


*No missing values were found and no duplicates, hence no cleaning is needed.*

### Outliers:
To identify outliers in a given column, using the Standard Deviation Method. It measures the amount of variation or dispersion in a set of values. 

Outliers are typically defined as data points that lie a certain number of standard deviations away from the mean.
- Defining numerical features by which outliners will be reviewed 
- Using mean() to find the mean of the dataset. This represents the central tendency of the data.
- Using std() to measure the average distance between each data point and the mean. 
- Threshold is set as 3, to identify outliers. Data points that fall outside this range are considered outliers. 
- Using a loop which collects outlier details.
- Printing the artist and track_name columns of the outliers, displaying them without the index.

In [57]:

numerical_features = [
    "energy",
    "danceability",
    "loudness",
    "acousticness",
    "speechiness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo",
    "duration_min",
]


mean = df[numerical_features].mean()
std_dev = df[numerical_features].std()
threshold = 3


outliers = (
    (df[numerical_features] < (mean - threshold * std_dev))
    | (df[numerical_features] > (mean + threshold * std_dev))
).any(axis=1)
df_outliers = df[outliers]
outlier_details = []


for feature in numerical_features:
    outlier_mask = (
        df_outliers[feature] < (mean[feature] - threshold * std_dev[feature])
    ) | (df_outliers[feature] > (mean[feature] + threshold * std_dev[feature]))

   
    outlier_info = df_outliers[outlier_mask][["artist", "track_name"]].copy()
    outlier_info["feature"] = feature
    outlier_info["outlier_value"] = df_outliers[outlier_mask][feature]
    outlier_info["deviation"] = df_outliers[outlier_mask][feature] - mean[feature]
    outlier_info["mean"] = mean[feature]
    outlier_info["std_dev"] = std_dev[feature]

    outlier_details.append(outlier_info)

outlier_summary = pd.concat(outlier_details)

print(
    outlier_summary[
        [
            "artist",
            "track_name",
            "feature",
            "outlier_value",
            "deviation",
            "mean",
            "std_dev",
        ]
    ].to_string(index=False)
)

         artist                 track_name          feature  outlier_value  deviation      mean  std_dev
  Billie Eilish        everything i wanted         loudness        -14.454  -8.228100 -6.225900 2.349744
         Future Life Is Good (feat. Drake)      speechiness          0.487   0.362842  0.124158 0.116836
  Billie Eilish        everything i wanted instrumentalness          0.657   0.641038  0.015962 0.094312
    Roddy Ricch                    The Box         liveness          0.790   0.593448  0.196552 0.176610
Black Eyed Peas  RITMO (Bad Boys For Life)         liveness          0.792   0.595448  0.196552 0.176610
   Travis Scott                 SICKO MODE     duration_min          5.210   1.876600  3.333400 0.566019


 *Outliers can often represent significant artistic or production choices, which are typical in the music industry. The presence of outliers in attributes like loudness or danceability could reveal important characteristics about the top tracks in the chart (e.g., higher or lower energy levels). Thus, removing them might lead to the loss of valuable insights about diverse musical trends, therefore outliers will not be removed.*

## **Data analysis**

### How many observations are there in this dataset and how many features this dataset has?
To count all observations and features shape is used 0 for rows and to calculate the number of columns that are not 'Unnamed: 0'

In [58]:
num_observations = df.shape[0]
print("Number of observations:", num_observations)
num_features = len(df.columns[df.columns != 'Unnamed: 0']) 
print("Number of features:", num_features)

Number of observations: 50
Number of features: 16


### Which of the features are categorical and which are numeric?
Dtypes are used to write down all the types in the columns and filtered to Objects or Not objects and converted to Python list with tolist to have a list instead of a column. 

In [59]:
categorical_features = df.dtypes[df.dtypes == "object"].index.tolist()
numerical_features = df.dtypes[df.dtypes != "object"].index.tolist()

print("Categorical features:", categorical_features)
print("Numerical features:", numerical_features)

Categorical features: ['artist', 'album', 'track_name', 'track_id', 'genre']
Numerical features: ['Unnamed: 0', 'energy', 'danceability', 'key', 'loudness', 'acousticness', 'speechiness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_min']


### Are there any artists that have more than 1 popular track? If yes, which and how many?
Group by is used to group by artist name, where is used to drop artists which have 1 track within the chart

In [60]:
number_of_tracks = df.groupby("artist").size()
number_of_tracks.where(number_of_tracks > 1).dropna()

artist
Billie Eilish    3.0
Dua Lipa         3.0
Harry Styles     2.0
Justin Bieber    2.0
Lewis Capaldi    2.0
Post Malone      2.0
Travis Scott     3.0
dtype: float64

### Who was the most popular artist?
The most popular artist is deemed to be the one who has most of tracks and albums within the chart. 

Loop is also used for situation if there are more than one artist with same number of tracks and albums.

Separator for better readability is used. 

In [61]:
number_of_albums = df.groupby("artist")["album"].nunique()

tracks_and_albums = number_of_albums + number_of_tracks

max_count = tracks_and_albums.max()
popular_artists = tracks_and_albums[tracks_and_albums == max_count].index


for artist in popular_artists:
    artist_albums = number_of_albums[artist]
    artist_tracks = number_of_tracks[artist]

    print(f"Artist: {artist}")
    print(f"Total albums: {artist_albums}")
    print(f"Total tracks: {artist_tracks}\n")

    artist_data = df[df["artist"] == artist][["album", "track_name"]]

    print("Albums and Tracks:" "\n", artist_data.to_string(index=False))
    print("\n" + "-" * 50 + "\n")  # Separator for better readability

Artist: Billie Eilish
Total albums: 3
Total tracks: 3

Albums and Tracks:
                                    album           track_name
                     everything i wanted  everything i wanted
WHEN WE ALL FALL ASLEEP, WHERE DO WE GO?              bad guy
                    lovely (with Khalid) lovely (with Khalid)

--------------------------------------------------

Artist: Travis Scott
Total albums: 3
Total tracks: 3

Albums and Tracks:
                           album          track_name
            HIGHEST IN THE ROOM HIGHEST IN THE ROOM
Birds In The Trap Sing McKnight          goosebumps
                     ASTROWORLD          SICKO MODE

--------------------------------------------------



### How many artists and albums in total have their songs in the top 50?
Unique is used to filter only unique artists and size to see the number

In [62]:
total_albums = df["album"].unique().size
total_artists = df["artist"].unique().size

print(f"Total artist number in the chart:", total_artists)
print(f"Total album number in the chart:", total_albums)

Total artist number in the chart: 40
Total album number in the chart: 45


### Are there any albums that have more than 1 popular track? If yes, which and how many?
Group by is used to group by album name, where used to drop albums which have 1 track within the chart

In [63]:
number_of_tracks_albums = df.groupby("album").size()
number_of_tracks_albums.where(number_of_tracks_albums > 1).dropna()

album
Changes                 2.0
Fine Line               2.0
Future Nostalgia        3.0
Hollywood's Bleeding    2.0
dtype: float64

### Which tracks have a danceability score above 0.7 and what tracks have danceability score below 0.4?
Loc is used to filter by danceability to be higher than 0.7 and to be below 0.4, then sort is used to sort the results. 

Head is used to display top result. 

Separator is added for better readability. 

In [64]:
danceability_high = df.loc[
    df["danceability"] > 0.7, ["track_name", "danceability"]
].sort_values(by="danceability", ascending=False)

danceability_low = df.loc[
    df["danceability"] < 0.4, ["track_name", "danceability"]
].sort_values(by="danceability", ascending=False)

print("Danceability above 0.7:" "\n", danceability_high.head())
print("\n" + "-" * 50 + "\n") 
print("Danceability below 0.4:" "\n", danceability_low.head())

Danceability above 0.7:
                           track_name  danceability
27   WAP (feat. Megan Thee Stallion)         0.935
2                            The Box         0.896
39                           Ride It         0.880
28                       Sunday Best         0.878
33  Supalonely (feat. Gus Dapperton)         0.862

--------------------------------------------------

Danceability below 0.4:
               track_name  danceability
44  lovely (with Khalid)         0.351


### Which tracks have their loudness above -5 and which have their loudness below -8?
loc is used to filter by danceability to be higher than -5 and to be below -8, then sort is used to sort the results. 

Head is used to display top result. 

Separator for better readability is used. 

In [65]:
loudness_high = df.loc[df["loudness"] > -5, ["track_name", "loudness"]].sort_values(
    by="loudness", ascending=False
)

loudness_low = df.loc[df["loudness"] < -8, ["track_name", "loudness"]].sort_values(
    by="loudness", ascending=False
)

print("Loudness above -5:" "\n", loudness_high.head())
print("\n" + "-" * 50 + "\n")
print("Loudness below -8:" "\n", loudness_low.head())

Loudness above -5:
         track_name  loudness
10            Tusa    -3.280
40      goosebumps    -3.370
31  Break My Heart    -3.434
38           Hawái    -3.454
12         Circles    -3.497

--------------------------------------------------

Loudness below -8:
                           track_name  loudness
20  Savage Love (Laxed - Siren Beat)    -8.520
8                            Falling    -8.756
36               HIGHEST IN THE ROOM    -8.764
7   death bed (coffee for your head)    -8.765
15                      Toosie Slide    -8.820


### Which track is the longest and which is the shortest?
loc is used to filter by idxmax and idxmin to be the longest and shortest.

Separator for better readability is used. 

In [66]:
longest = df.loc[df["duration_min"].idxmax(), ["artist", "track_name", "duration_min"]]
shortest = df.loc[df["duration_min"].idxmin(), ["artist", "track_name", "duration_min"]]

print(f"Longest track information:" "\n", longest)
print("\n" + "-" * 50 + "\n") 
print(f"Shortest track information:" "\n", shortest)

Longest track information:
 artist          Travis Scott
track_name        SICKO MODE
duration_min            5.21
Name: 49, dtype: object

--------------------------------------------------

Shortest track information:
 artist                        24kGoldn
track_name      Mood (feat. iann dior)
duration_min                      2.34
Name: 23, dtype: object


### Which genre is the most popular?
The most popular genre is deemed to be the one who has most of tracks within the chart. 
str.split is used to split genra, since one track can have several genre. 
Explode is used to have each genre in separate line. 
Loop is used for situation if there are more than one genre with same number of tracks. 

In [67]:
df["genres"] = df["genre"].str.split("/")
df_exploded = df.explode("genres")
genre_counts = df_exploded.groupby("genres").size()
max_count = genre_counts.max()
top_genres = genre_counts[genre_counts == max_count]

print("Top genres:")
for genre, count in top_genres.items():
    print(f"Genre: {genre}, Count: {count}")

Top genres:
Genre: Hip-Hop, Count: 15
Genre: Pop, Count: 15


### Which genres have just one song on the top 50?
since we have genre_counts, we can simple extract values of 1 and create a loop to show only genre name

In [68]:
rare_genres = genre_counts[genre_counts == 1]
for genre in rare_genres.index:
    print(genre)

Chamber pop
Dance-pop
Disco
Disco-pop
Dreampop
Hip-Hop alternative
Nu-disco
Pop rap
Soft Rock
Trap
experimental
reggaeton


### How many genres in total are represented in the top 50?
we have divided genres in df_exploded, we will count unique to have only unique number

In [69]:
total_genre = df_exploded["genres"].nunique()
print(f"Total number of genre in top 50: ", total_genre)

Total number of genre in top 50:  22


### Which features are strongly positively correlated?
Only numerical features are needed, therefore we need to define what are they in the begging to disregard the unnamed and key as they based on the logic cannot be correlated, after the the correlation matrix is created with corr().
Loop is used to check for repetititve information and adds values which are above threshold to the list which is used to generate the results.

In [70]:
correlation_matrix = df[numerical_features].corr()
print(correlation_matrix)

                  Unnamed: 0    energy  danceability       key  loudness  \
Unnamed: 0          1.000000  0.030381     -0.176321 -0.052844  0.034935   
energy              0.030381  1.000000      0.152552  0.062428  0.791640   
danceability       -0.176321  0.152552      1.000000  0.285036  0.167147   
key                -0.052844  0.062428      0.285036  1.000000 -0.009178   
loudness            0.034935  0.791640      0.167147 -0.009178  1.000000   
acousticness       -0.036557 -0.682479     -0.359135 -0.113394 -0.498695   
speechiness         0.095790  0.074267      0.226148 -0.094965 -0.021693   
instrumentalness   -0.003126 -0.385515     -0.017706  0.020802 -0.553735   
liveness           -0.063216  0.069487     -0.006648  0.278672 -0.069939   
valence            -0.034159  0.393453      0.479953  0.120007  0.406772   
tempo               0.081289  0.075191      0.168956  0.080475  0.102097   
duration_min        0.309557  0.080290     -0.034681 -0.005078  0.063487   

           

In [71]:
strong_positive_threshold = 0.6
strongly_positive_correlated = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i):
        if correlation_matrix.iloc[i, j] > strong_positive_threshold:
            colname = correlation_matrix.columns[i]
            rowname = correlation_matrix.index[j]
            strongly_positive_correlated.append(
                (rowname, colname, correlation_matrix.iloc[i, j])
            )

print("Strongly Positively Correlated Features:")
for feature in strongly_positive_correlated:
    print(f"{feature[0]} and {feature[1]}: {feature[2]}")

Strongly Positively Correlated Features:
energy and loudness: 0.7916395653045617


*Meaning, if the track is louder the more energy it has*

### Which features are strongly negatively correlated?
Loop is used to check for repetititve information and adds values which are lower the threshold to the list which is used to generate the results.

In [72]:
strong_negative_threshold = -0.6
strongly_negative_correlated = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i):
        if correlation_matrix.iloc[i, j] < strong_negative_threshold:
            colname = correlation_matrix.columns[i]
            rowname = correlation_matrix.index[j]
            strongly_negative_correlated.append(
                (rowname, colname, correlation_matrix.iloc[i, j])
            )
print("\nStrongly Negatively Correlated Features:")
for feature in strongly_negative_correlated:
    print(f"{feature[0]} and {feature[1]}: {feature[2]}")


Strongly Negatively Correlated Features:
energy and acousticness: -0.6824785203241528


*Meaning that if the aucusticness level is higher, the it will be less energetic*

### Which features are not correlated?
Loop is used to check for repetititve information and adds values which are lower the threshold to the list which is used to generate the results.

In [73]:
not_correlated = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i):
        if (
            abs(correlation_matrix.iloc[i, j]) < 0.1
        ):
            colname = correlation_matrix.columns[i]
            rowname = correlation_matrix.index[j]
            not_correlated.append((rowname, colname, correlation_matrix.iloc[i, j]))
print("\nFeatures Not Correlated:")

for feature in not_correlated:
    print(f"{feature[0]} and {feature[1]}: {feature[2]}")


Features Not Correlated:
Unnamed: 0 and energy: 0.030381062224497258
Unnamed: 0 and key: -0.05284389337266977
energy and key: 0.062428155998836
Unnamed: 0 and loudness: 0.034935308378584266
key and loudness: -0.009178410631968104
Unnamed: 0 and acousticness: -0.036557103192611747
Unnamed: 0 and speechiness: 0.09578967497742871
energy and speechiness: 0.07426734162813511
key and speechiness: -0.09496505735843172
loudness and speechiness: -0.021692935459647147
Unnamed: 0 and instrumentalness: -0.0031261891618863643
danceability and instrumentalness: -0.01770638521729678
key and instrumentalness: 0.020802356350748005
speechiness and instrumentalness: 0.028948017426321342
Unnamed: 0 and liveness: -0.06321551035150103
energy and liveness: 0.0694873918471837
danceability and liveness: -0.006648475599485623
loudness and liveness: -0.06993949725852708
instrumentalness and liveness: -0.0870339124461283
Unnamed: 0 and valence: -0.03415920941882543
speechiness and valence: 0.05386720274013534
li

*Meaning this features have low dependencies on each other*

### How does the loudness score and acousticness score compare between Pop, Hip-Hop/Rap, Dance/Electronic, and Alternative/Indie genres?
Features and genres lists are created, grouped by genras and agregation with numpu is used to have mean, max and min.

In [74]:
selected_features = ["danceability", "loudness", "acousticness"]
selected_genres = ["Pop", "Hip-Hop/Rap", "Dance/Electronic", "Alternative/Indie"]
df.groupby(["genre"])[selected_features].aggregate([np.mean, np.max, np.min]).loc[
    selected_genres
]

danceability                loudness                 \
                          mean   amax   amin      mean   amax    amin   
genre                                                                   
Pop                   0.677571  0.806  0.464 -6.460357 -3.280 -14.454   
Hip-Hop/Rap           0.765538  0.896  0.598 -6.917846 -3.370  -8.820   
Dance/Electronic      0.755000  0.880  0.647 -5.338000 -3.756  -7.567   
Alternative/Indie     0.661750  0.862  0.459 -5.421000 -4.746  -6.401   

                  acousticness                  
                          mean   amax     amin  
genre                                           
Pop                   0.323843  0.902  0.02100  
Hip-Hop/Rap           0.188741  0.731  0.00513  
Dance/Electronic      0.099440  0.223  0.01370  
Alternative/Indie     0.583500  0.751  0.29100

*By comparing mean in selected genre we can see:*
- *The most danceble genre is Hip-Hop/Rap, least is Alternative/Indie*
- *The loundest is Dance/Electronic, least Hip-Hop/Rap*
- *Most acustic is alternative/indie, lest Hip-Hop/Rap*


# **Summary of Observations**

The most popular artists are identified as Billie Eilish and Travis Scott, who both have 3 albums and 3 tracks within top 50. 

Columns are defined to see only relevant columns.

It is suggested to review this tracks deeper.

In [75]:
artists_of_interest = ["Billie Eilish", "Travis Scott"]
df_popular_artists = df[df["artist"].isin(artists_of_interest)]

display_columns = [
    "artist",
    "track_name",
    "energy",
    "danceability",
    "loudness",
    "acousticness",
    "speechiness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo",
    "duration_min",
    "genre",
]

print(df_popular_artists[display_columns].to_string(index=False))

       artist           track_name  energy  danceability  loudness  acousticness  speechiness  instrumentalness  liveness  valence   tempo  duration_min        genre
Billie Eilish  everything i wanted   0.225         0.704   -14.454       0.90200       0.0994          0.657000     0.106   0.2430 120.006          4.09          Pop
Billie Eilish              bad guy   0.425         0.701   -10.965       0.32800       0.3750          0.130000     0.100   0.5620 135.128          3.23  Electro-pop
 Travis Scott  HIGHEST IN THE ROOM   0.427         0.598    -8.764       0.05460       0.0317          0.000006     0.210   0.0605  76.469          2.93  Hip-Hop/Rap
 Travis Scott           goosebumps   0.728         0.841    -3.370       0.08470       0.0484          0.000000     0.149   0.4300 130.049          4.06  Hip-Hop/Rap
Billie Eilish lovely (with Khalid)   0.296         0.351   -10.109       0.93400       0.0333          0.000000     0.095   0.1200 115.284          3.34  Chamber pop
 Tra

### **Billie Eilish (Pop Genre):**

**Loudness:** Billie Eilish's tracks tend to be quieter than the average loudness for the Pop genre. This suggests that softer, more subtle tracks can be popular in this genre.

**Danceability:** Two out of three tracks have above-average danceability, indicating that while her tracks are quieter, they still encourage dancing.

**Acousticness:** All three tracks have acousticness close to or significantly above the mean, suggesting that incorporating acoustic elements can contribute to popularity in the Pop genre.

### **Travis Scott (Hip-Hop/Rap Genre):**

**Loudness:** Travis Scott's tracks are louder than the average for the Hip-Hop/Rap genre, indicating that louder tracks may be more popular in this genre.

**Danceability:** His tracks are more danceable than the average, suggesting that danceability is an important factor in the popularity of Hip-Hop/Rap tracks.

**Acousticness:** His tracks have lower acousticness than the average, indicating that less acoustic and more electronic or produced sounds may be favored in this genre.

### **Conclusions**

**Pop Genre:** To achieve popularity, tracks in the Pop genre may benefit from being quieter, danceable, and incorporating acoustic elements. This combination can differentiate them from other artists and appeal to listeners who appreciate a more nuanced sound.

**Hip-Hop/Rap Genre:** Popularity in the Hip-Hop/Rap genre may be associated with louder tracks that are danceable and have lower acousticness. This suggests a preference for energetic, produced sounds that encourage movement and engagement.